# Laboratorio 0: Intro Gradio & BeautifulSoup

En este primer acercamiento, trabajaremos con dos herramientas fundamentales. Por un lado, **Gradio**, una librería que permite construir interfaces web interactivas en pocos minutos. Es ideal para probar modelos o funciones sin necesidad de escribir código front-end complejo. Por otro lado, presentaremos **Beautiful Soup**, una de las librerías más clásicas y robustas en Python para analizar (parsear) documentos HTML y extraer información específica de sus etiquetas.

El objetivo de este cuaderno es familiarizarse con la sintaxis básica de ambas librerías por separado, para luego comprender cómo integrarlas en flujos de trabajo más grandes.

In [ ]:
# Instalamos la librería Gradio de forma silenciosa (-q)
!pip install gradio -q

In [ ]:
# Importamos la librería y le damos el alias 'gr' para que llames a sus métodos más fácil
import gradio as gr

## Creando la primera interfaz
Notemos que una interfaz de Gradio requiere siempre tres elementos principales: una función de Python que procese los datos, un componente de entrada (`inputs`) y un componente de salida (`outputs`).

In [ ]:
# Definimos una función simple que recibe un nombre y devuelve un saludo
def bienvenida(name):
    return "Hola " + name

# Inciamos la interfaz conectando la función con entradas y salidas de texto
bienvenido = gr.Interface(fn=bienvenida, inputs="text", outputs="text")

# Lanzamos la aplicación web
bienvenido.launch()

Recordá que podés usar componentes más avanzados, como deslizadores (sliders), para recibir parámetros numéricos de manera simplificada.

In [ ]:
# Esta función recibe un texto y un número entero (intensidad)
def saludo(nombre, intensidad):
    return "Hola, " + nombre + "!" * int(intensidad)

# Le pasamos una lista a las entradas: un cuadro de texto y un slider
demo = gr.Interface(
    fn=saludo,
    inputs=["text", "slider"],
    outputs=["text"],
)

demo.launch()

También es posible personalizar los componentes manualmente, instanciando sus clases específicas (por ejemplo, pasándole `label` y `placeholder` a un `gr.Textbox`).

In [ ]:
def greet(name):
    return "Hello " + name

# Configuramos la clase Textbox a gusto
textbox = gr.Textbox(label="Tipeá tu nombre acá:", placeholder="Juan Pérez", lines=2)

gr.Interface(fn=greet, inputs=textbox, outputs="text").launch()

---
## Explorando Beautiful Soup
Beautiful Soup es un parser analítico. Toma un texto HTML en formato crudo y lo convierte en un árbol de objetos de Python con el que resulta mucho más intuitivo interactuar.

In [ ]:
# Importamos la clase estructuradora principal
from bs4 import BeautifulSoup

In [ ]:
# Preparamos un string crudo simulando una página web cortita
html = """
<html>
<head>
  <title>Mi Página de Ejemplo</title>
</head>
<body>
  <h1>Título Principal</h1>
  <p>Este es un párrafo.</p>
  <a href="https://www.ejemplo.com">Enlace a Ejemplo</a>
</body>
</html>
"""

Al instanciar el objeto `BeautifulSoup`, le indicamos explícitamente qué analizador utilizar. El analizador clásico por defecto de Python es `html.parser`.

In [ ]:
# Transformamos el texto plano (string) a objeto navgeable (soup)
soup = BeautifulSoup(html, 'html.parser')
print(soup)

La utilización de atajos simplifica el trabajo. La librería permite usar la notación de punto (`.`) para navegar directamente hacia las etiquetas estructurales comunes (`soup.etiqueta`). Para extraer la información textual, utilizamos la propiedad `.text`.

In [ ]:
titulo = soup.title
print("Etiqueta completa:", titulo)       # Muestra todo el tag HTML <title>
print("Texto interno:", titulo.text)      # Muestra únicamente tu información útil

In [ ]:
parrafo = soup.p
print("Etiqueta completa:", parrafo)
print("Texto interno:", parrafo.text)

Si el servidor web devuelve el código comprimido en una sola línea (minificado), podés utilizar el método `.prettify()`. Este método reordena la estructura y añade tabulaciones para facilitar la lectura de las etiquetas en la terminal.

In [ ]:
html_compacto = "<html><head><title>Mi Página</title></head><body><h1>Hola</h1></body></html>"
soup_compacto = BeautifulSoup(html_compacto, 'html.parser')

# Acá le damos forma presentable
print(soup_compacto.prettify())

Por último, abordamos los **atributos**. Una técnica fundamental del web scraping consiste en extraer el enlace de destino de una etiqueta `<a>` (su atributo `href`). Beautiful Soup gestiona los atributos del mismo modo en que Python administra las claves de un diccionario.

In [ ]:
html_enlace = """
<a href="https://www.ejemplo.com" class="enlace-principal">Ir a Ejemplo</a>
"""
soup_enlace = BeautifulSoup(html_enlace, 'html.parser')
enlace = soup_enlace.a

# Leemos los atributos accediendo con corchetes e indicando la 'clave'
print("URL extraída (href):", enlace['href'])   
print("Clase extraída:", enlace['class'])  

---
# Consolidación y Repaso

## Glosario Acotado
*   **Interfaz Gráfica (UI):** Entorno visual interactivo creado como capa superior al código subyacente, diseñado para facilitar la interacción del usuario sin requerir conocimientos de terminal o consola.
*   **Parsing (Análisis Sintáctico):** Función principal de la librería Beautiful Soup. Refiere al proceso algorítmico mediante el cual se transforma una cadena de texto estructurada plana en un árbol jerárquico de objetos navegables en Python.
*   **Atributo HTML:** Modificadores o propiedades adicionales incorporadas en una etiqueta (Tag), tales como identificadores únicos (`id`) o hipervínculos de destino (`href`).

## Preguntas de Autoevaluación

**1. ¿Por qué diferenciamos entre llamar a un objeto `soup.p` y recuperar su atributo `.text`?**
Llamar al objeto nativo devuelve la entidad HTML completa, incluyendo todo su marcado estructural. Durante un proceso de extracción de datos, el objetivo es capturar la información útil de la página. La propiedad `.text` filtra las etiquetas subyacentes, preservando únicamente la información visible por el usuario final.

**2. ¿Resulta una buena práctica utilizar el esquema de diccionario para extraer un atributo (por ejemplo, `['href']`) trabajando con listas dinámicas?**
Se desaconseja, debido a que referenciar una clave inexistente en un diccionario de Python generará una interrupción de ejecución por `KeyError`. Es una buena práctica técnica utilizar el método `.get('href')`, el cual captura la excepción y retorna el valor nulo (`None`) si el atributo buscado no forma parte del nodo.

**3. ¿Por qué razón enviaríamos múltiples componentes dentro de una lista hacia Gradio?**
Gradio mapea estrictamente los parámetros de entrada y salida con la firma de la función enlazada. Proveer componentes empaquetados en una lista permite diseñar flujos donde múltiples inputs representan variables correlacionales en funciones que demandan mayor cantidad de argumentos (como en evaluaciones matemáticas o predicciones multivariables).